# main.py — FastAPI 기반 반려동물 가상 피팅 API 서버

## 이 파일은 무엇인가요?

이 파일은 프로젝트의 **"정문(입구)"** 역할을 하는 **웹 API 서버**입니다.

외부 서비스(예: Spring Boot로 만든 메인 서버, 모바일 앱 등)에서 **"이 반려동물 사진에 이 옷을 입혀줘!"** 라는 요청(Request)을 보내면, 이 서버가 그 요청을 받아서 AI 파이프라인에게 작업을 시키고, 결과물을 돌려주는 역할을 합니다.

### 전체 흐름 요약

```
[외부 서버/앱] —(HTTP 요청)—> [이 파일: main.py (FastAPI 서버)]
                                    |
                                    ├─ 1. Firestore DB에 "처리 중" 상태 기록
                                    ├─ 2. 반려동물 사진 & 옷 사진을 URL에서 다운로드
                                    ├─ 3. vton_pipeline.py의 AI 파이프라인에게 작업 요청
                                    ├─ 4. AI가 만든 결과 이미지를 Google Cloud Storage에 업로드
                                    └─ 5. Firestore DB에 "완료" 상태 + 결과 URL 기록
                                    |
                            —(HTTP 응답)—> [외부 서버/앱]
```

### 사용된 기술 스택

| 기술 | 역할 |
|---|---|
| **FastAPI** | 파이썬 웹 프레임워크. HTTP 요청을 받고 응답을 보내는 서버를 만들어줍니다. |
| **Pydantic (BaseModel)** | 외부에서 들어오는 JSON 데이터의 형식을 검증합니다. (필수 필드 누락 방지 등) |
| **Firebase Admin SDK** | Google의 Firestore 데이터베이스에 접속하여 작업 상태를 읽고/쓰기 합니다. |
| **Google Cloud Storage (GCS)** | AI가 생성한 결과 이미지를 클라우드 저장소에 업로드합니다. |
| **vton_pipeline** | 실제 AI 모델이 동작하는 파이프라인 모듈 (별도 파일). |

## 1단계: 라이브러리 불러오기 (Import)

이 서버를 만들기 위해 필요한 외부 도구(라이브러리)들을 불러옵니다.

- `FastAPI`: 웹 서버 프레임워크. `@app.post()` 같은 데코레이터로 API 엔드포인트를 쉽게 만들 수 있습니다.
- `HTTPException`: API 요청 처리 중 에러가 발생했을 때, 클라이언트에게 에러 코드(500 등)와 메시지를 보내는 도구입니다.
- `BaseModel`: 외부에서 들어오는 JSON 데이터의 형태(어떤 필드가 있어야 하고, 타입은 무엇인지)를 미리 정의해두는 도구입니다. 잘못된 데이터가 들어오면 자동으로 에러를 반환해줍니다.
- `urllib.request`: 인터넷 URL에서 파일(이미지 등)을 다운로드하는 파이썬 기본 내장 모듈입니다.
- `firebase_admin`, `firestore`: Google Firebase에 접속하여 Firestore(NoSQL 데이터베이스)를 읽고 쓰는 도구입니다.
- `storage`: Google Cloud Storage(GCS)에 파일을 업로드/다운로드하는 도구입니다.
- `pipeline`: 우리가 만든 AI 파이프라인(`vton_pipeline.py`)에서 미리 생성해둔 파이프라인 인스턴스(객체)를 가져옵니다.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import urllib.request
import firebase_admin
from firebase_admin import firestore
from google.cloud import storage
from vton_pipeline import pipeline

## 2단계: Firebase 초기화 및 Firestore DB 연결

서버가 시작되자마자 Firebase에 접속합니다.

- `firebase_admin.initialize_app()`: Firebase 서비스를 초기화합니다. Google Cloud Run 같은 GCP 환경에서 실행하면, 별도의 인증 키 파일 없이도 **자동으로 인증**됩니다. (서비스 계정이 자동으로 연결됨)
- `db = firestore.client()`: Firestore 데이터베이스에 접속할 수 있는 클라이언트 객체를 만듭니다. 이 `db` 변수를 통해 데이터를 읽고 쓸 수 있습니다.

> **Firestore란?** Google이 제공하는 클라우드 NoSQL 데이터베이스입니다. JSON 형태의 문서(Document)를 컬렉션(Collection)으로 묶어 저장합니다. SQL처럼 테이블/행 구조가 아닌, `컬렉션 > 문서` 구조입니다.

In [ ]:
firebase_admin.initialize_app()
db = firestore.client()

## 3단계: Google Cloud Storage(GCS) 업로드 함수

AI가 생성한 결과 이미지(바이트 데이터)를 **Google Cloud Storage 버킷**에 업로드하고, 업로드된 이미지의 **공개 URL(웹 주소)을 반환**하는 함수입니다.

### 매개변수 설명
- `image_bytes`: AI가 생성한 결과 이미지의 바이트 데이터 (JPEG 형식)
- `destination_blob_name`: GCS 버킷 안에서의 저장 경로 및 파일명 (예: `results/test_user_001_output.jpg`)

### 동작 과정
1. `storage.Client()`로 GCS에 접속합니다.
2. `pet-fitting-images`라는 이름의 버킷(폴더 같은 개념)을 선택합니다.
3. 버킷 안에 `destination_blob_name` 경로로 이미지를 업로드합니다.
4. 업로드된 이미지의 공개 URL을 반환합니다.

> **버킷(Bucket)이란?** GCS에서 파일을 저장하는 최상위 컨테이너(폴더)입니다. 전 세계적으로 이름이 고유해야 합니다.

In [ ]:
def upload_to_gcs(image_bytes, destination_blob_name):
    bucket_name = "pet-fitting-images"
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_string(image_bytes, content_type='image/jpeg')
    return blob.public_url

## 4단계: FastAPI 앱 생성 및 요청 데이터 형식 정의

### FastAPI 앱 생성
`FastAPI()`를 호출하면 웹 서버 앱이 만들어집니다. `title`은 자동 생성되는 API 문서(Swagger UI)에 표시되는 제목입니다.

### FittingRequest 클래스 (요청 데이터 검증)
외부(Spring Boot 등)에서 이 서버로 요청을 보낼 때 **반드시 포함해야 하는 JSON 필드 3개**를 정의합니다.

| 필드명 | 타입 | 설명 |
|---|---|---|
| `user_id` | 문자열(str) | 요청을 보낸 사용자의 고유 ID |
| `pet_image_url` | 문자열(str) | 반려동물 사진이 저장된 인터넷 URL |
| `cloth_image_url` | 문자열(str) | 입히고 싶은 옷 사진이 저장된 인터넷 URL |

만약 외부에서 이 3개 필드 중 하나라도 빠뜨리거나, 문자열이 아닌 다른 타입(숫자 등)을 보내면 FastAPI가 자동으로 **422 에러(Unprocessable Entity)**를 반환합니다.

### download_image_as_bytes 함수
인터넷 URL로부터 이미지를 다운로드하여 **바이트(bytes)** 형태로 반환합니다. `User-Agent` 헤더를 설정하는 이유는, 일부 서버가 봇(자동화 프로그램)의 접근을 차단하는 것을 우회하기 위함입니다.

In [ ]:
app = FastAPI(title="Pet VTON AI Worker")

class FittingRequest(BaseModel):
    user_id: str
    pet_image_url: str
    cloth_image_url: str

def download_image_as_bytes(url: str) -> bytes:
    """URL에서 이미지를 바이트 형태로 다운로드합니다."""
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response:
        return response.read()

## 5단계: 핵심 API 엔드포인트 — `/api/v1/fit` (POST)

이 함수가 **전체 서버의 핵심 로직**입니다. 외부에서 `POST /api/v1/fit` 요청이 들어오면 아래 순서대로 실행됩니다.

### 처리 순서 (5단계)

| 순서 | 작업 | 상세 설명 |
|---|---|---|
| **1** | Firestore에 "처리 중" 기록 | `fittings` 컬렉션에 해당 사용자 문서의 상태를 `processing`으로 설정합니다. `merge=True`를 사용하여 기존 데이터가 있으면 덮어쓰지 않고 해당 필드만 업데이트합니다. |
| **2** | 이미지 다운로드 | 전달받은 URL에서 반려동물 사진(`pet_bytes`)과 옷 사진(`cloth_bytes`)을 바이트 형태로 다운로드합니다. |
| **3** | AI 파이프라인 실행 | `vton_pipeline.py`에서 가져온 `pipeline` 객체의 `generate_fitting()` 메서드를 호출합니다. 이 단계에서 SAM, Stable Diffusion 등 무거운 AI 연산이 수행되며 **가장 오래 걸립니다.** |
| **4** | 결과를 GCS에 업로드 | AI가 만든 합성 이미지를 GCS 버킷에 업로드하고, 결과 이미지의 공개 URL을 받아옵니다. |
| **5** | Firestore에 "완료" 기록 | 상태를 `done`으로 변경하고, 결과 이미지 URL(`resultUrl`)을 함께 저장합니다. 프론트엔드나 메인 서버는 이 상태 변화를 감지하여 사용자에게 결과를 보여줄 수 있습니다. |

### 에러 처리
`try-except`로 감싸서, 위 5단계 중 어디에서든 에러가 발생하면 **HTTP 500 에러(서버 내부 오류)**와 함께 에러 메시지를 클라이언트에게 반환합니다.

### 성공 시 반환하는 JSON 형식
```json
{
    "status": "success",
    "message": "Fitting completed successfully.",
    "result_url": "https://storage.googleapis.com/pet-fitting-images/results/..."
}
```

In [ ]:
@app.post("/api/v1/fit")
async def process_fitting(req: FittingRequest):
    try:
        # 1. Firestore에 처리 시작 상태 기록
        db.collection('fittings').document(req.user_id).set({
            "status": "processing"
        }, merge=True)

        # 2. 이미지 다운로드 (스토리지 URL -> 바이트)
        pet_bytes = download_image_as_bytes(req.pet_image_url)
        cloth_bytes = download_image_as_bytes(req.cloth_image_url)

        # 3. AI 파이프라인 실행 (무거운 작업)
        result_bytes = pipeline.generate_fitting(pet_bytes, cloth_bytes)

        # 4. 결과 이미지를 GCS에 업로드
        result_url = upload_to_gcs(result_bytes, f"results/{req.user_id}_output.jpg")

        # 5. Firestore에 완료 상태 + 결과 URL 기록
        db.collection('fittings').document(req.user_id).update({
            "status": "done",
            "resultUrl": result_url
        })
        
        return {
            "status": "success",
            "message": "Fitting completed successfully.",
            "result_url": result_url
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

## 6단계: 헬스 체크 엔드포인트 — `/health` (GET)

이 엔드포인트는 **서버가 살아있는지(정상 동작 중인지) 확인**하기 위한 용도입니다.

- Google Cloud Run은 주기적으로 이 엔드포인트에 `GET` 요청을 보내서, 서버가 응답하면 "정상", 응답이 없으면 "장애 발생"으로 판단합니다.
- 단순히 `{"status": "ok"}`만 반환하면 됩니다. 실제 AI 연산은 수행하지 않습니다.
- 이런 패턴을 **Health Check** 또는 **Liveness Probe**라고 부릅니다.

In [ ]:
@app.get("/health")
def health_check():
    """Cloud Run이 서버가 살아있는지 확인하는 엔드포인트"""
    return {"status": "ok"}